# ZenFit Meal Classifier — Colab GPU Workflow
Run cells in order. Every expensive or state-changing action is opt-in. Dataset and model state are always revalidated from disk.

## 1. Runtime verification

In [ ]:
import os, sys, json, platform, subprocess, hashlib, shutil, time
from pathlib import Path
print({'python':sys.version,'platform':platform.platform(),'cwd':os.getcwd()})
IN_COLAB='google.colab' in sys.modules
print('Google Colab runtime:',IN_COLAB)

## 2. Repository setup

In [ ]:
REPO_PATH=Path(os.getenv('ZENFIT_REPO_PATH','/content/ZenFit'))
REPO_URL=os.getenv('ZENFIT_REPO_URL','')
if not (REPO_PATH/'backend/training').is_dir():
    if not REPO_URL: raise FileNotFoundError('Set ZENFIT_REPO_PATH or ZENFIT_REPO_URL')
    subprocess.run(['git','clone',REPO_URL,str(REPO_PATH)],check=True)
BACKEND_PATH=REPO_PATH/'backend'; os.chdir(BACKEND_PATH)
if str(BACKEND_PATH) not in sys.path: sys.path.insert(0,str(BACKEND_PATH))
print('Repository:',REPO_PATH)
# For private Git, use a short-lived Colab secret/environment credential helper. Never embed or print a token.

## 3. Dependency setup

In [ ]:
INSTALL_DEPS=False
if INSTALL_DEPS: subprocess.run([sys.executable,'-m','pip','install','-r','requirements-training.txt'],check=True)
import torch, torchvision, numpy, pandas, sklearn, PIL
from PIL import Image
print({'torch':torch.__version__,'torchvision':torchvision.__version__,'numpy':numpy.__version__,'pandas':pandas.__version__,'sklearn':sklearn.__version__,'Pillow':PIL.__version__})

## 4. GPU verification

In [ ]:
print('torch.cuda.is_available():',torch.cuda.is_available()); print('CUDA version:',torch.version.cuda)
if torch.cuda.is_available():
    props=torch.cuda.get_device_properties(0); print('GPU name:',torch.cuda.get_device_name(0)); print('Total GPU memory (GiB):',round(props.total_memory/2**30,2)); print('Allocated (GiB):',round(torch.cuda.memory_allocated()/2**30,3)); print('Reserved (GiB):',round(torch.cuda.memory_reserved()/2**30,3))
else: print('CUDA is unavailable. Dataset checks may run, but training cells will fail before training.')

## 5. Paths and configuration

In [ ]:
from collections import Counter
LOCAL_ROOT=Path(os.getenv('ZENFIT_COLAB_ROOT','/content/zenfit-work'))
RAW_ROOT=LOCAL_ROOT/'data/raw/kaggle'; DATASET=LOCAL_ROOT/'data/training/indian_food_v2'; REPORTS=LOCAL_ROOT/'reports'; MODELS=LOCAL_ROOT/'models/indian_food'; PACKAGES=LOCAL_ROOT/'artifacts'
raw=RAW_ROOT/'food_image_classification'/'Food Classification dataset'; manifest_path=DATASET/'split_manifest.json'; DRIVE_ROOT=None
for item in (RAW_ROOT,REPORTS,MODELS,PACKAGES): item.mkdir(parents=True,exist_ok=True)
IMAGE_SUFFIXES={'.jpg','.jpeg','.png','.webp','.bmp'}
def validate_prepared_dataset(dataset):
    dataset=Path(dataset); splits=('train','val','test'); errors=[]; manifest=None
    actual={s:sum(1 for p in (dataset/s).rglob('*') if p.is_file() and p.suffix.lower() in IMAGE_SUFFIXES) if (dataset/s).is_dir() else 0 for s in splits}
    for s in splits:
        if not (dataset/s).is_dir(): errors.append(f'{s} directory is missing')
        elif actual[s]==0: errors.append(f'{s} split is empty')
    path=dataset/'split_manifest.json'
    if not path.is_file(): errors.append('split_manifest.json is missing')
    else:
        try: manifest=json.loads(path.read_text())
        except (OSError,json.JSONDecodeError) as exc: errors.append(f'split_manifest.json cannot be parsed: {exc}')
    expected={s:0 for s in splits}
    if manifest is not None:
        files=manifest.get('files')
        if not isinstance(files,list) or not files: errors.append('manifest files list is missing or empty')
        else:
            for i,row in enumerate(files):
                value=row.get('path') if isinstance(row,dict) else None; parts=Path(value).parts if isinstance(value,str) and value else ()
                if not parts or parts[0] not in splits: errors.append(f'manifest file {i} has an invalid split path')
                else: expected[parts[0]]+=1
            for s in splits:
                if expected[s]!=actual[s]: errors.append(f'{s} count mismatch: manifest={expected[s]}, actual={actual[s]}')
    return {'valid':not errors,'manifest':manifest if not errors else None,'manifest_counts':expected,'actual_counts':actual,'errors':errors}
print({'raw':str(raw),'prepared':str(DATASET),'reports':str(REPORTS),'models':str(MODELS)})

## 6. Kaggle authentication

In [ ]:
def configure_kaggle_auth():
    token=os.getenv('KAGGLE_API_TOKEN')
    if not token and IN_COLAB:
        from google.colab import userdata
        token=userdata.get('KAGGLE_API_TOKEN')
    if token: os.environ['KAGGLE_API_TOKEN']=token
    if not (token or (Path.home()/'.kaggle/kaggle.json').is_file()): raise RuntimeError('Configure Kaggle via a Colab secret, environment variable, or secure kaggle.json')
    print('Kaggle credentials are configured (secret not displayed).')

## 7. Dataset acquisition

In [ ]:
DOWNLOAD_DATASET=False
if DOWNLOAD_DATASET:
    configure_kaggle_auth(); subprocess.run([sys.executable,'training/download_kaggle_datasets.py','--dataset','food_image_classification','--root',str(RAW_ROOT)],check=True)
print('Raw dataset available:',raw.is_dir())

## 8. Dataset validation and preparation

In [ ]:
if not raw.is_dir(): raise FileNotFoundError(f'Raw dataset is missing: {raw}. Run dataset acquisition first.')
status=validate_prepared_dataset(DATASET)
if status['valid']:
    manifest=status['manifest']; print('Prepared dataset is valid. Reusing existing dataset.')
else:
    manifest=None
    if DATASET.exists(): print('Incomplete prepared dataset detected. Regenerating.'); shutil.rmtree(DATASET)
    else: print('Prepared dataset not found. Creating train/val/test splits.')
    subprocess.run([sys.executable,'training/prepare_class_labeled_v2.py','--raw',str(raw),'--output',str(DATASET),'--reports',str(REPORTS)],check=True)
    status=validate_prepared_dataset(DATASET)
    if not status['valid']: manifest=None; raise RuntimeError('Dataset preparation finished but validation failed: '+'; '.join(status['errors']))
    manifest=json.loads(manifest_path.read_text()); print('Prepared dataset created and validated.')
print({'manifest_counts':status['manifest_counts'],'actual_counts':status['actual_counts']})

## 9. Split verification

In [ ]:
status=validate_prepared_dataset(DATASET)
if not status['valid']: manifest=None; raise RuntimeError('Prepared dataset is invalid: '+'; '.join(status['errors']))
manifest=status['manifest']; print('Manifest counts:',status['manifest_counts']); print('Actual file counts:',status['actual_counts']); assert status['manifest_counts']==status['actual_counts']
print('Class distribution:',json.dumps({n:{k:v for k,v in row.items() if k in ('total','train','val','test')} for n,row in manifest.get('classes',{}).items()},indent=2))

## 10. Duplicate and leakage verification

In [ ]:
status=validate_prepared_dataset(DATASET)
if not status['valid']: raise RuntimeError('Prepared dataset is invalid: '+'; '.join(status['errors']))
manifest=status['manifest']; missing=[i for i,row in enumerate(manifest['files']) if not row.get('sha256')]
if missing: raise RuntimeError(f'Manifest is missing sha256 for {len(missing)} files')
hash_splits={}
for row in manifest['files']: hash_splits.setdefault(row['sha256'],set()).add(Path(row['path']).parts[0])
duplicates=len(manifest['files'])-len(hash_splits); leaks={h:sorted(s) for h,s in hash_splits.items() if len(s)>1}
if duplicates: raise RuntimeError(f'Duplicate SHA256 values detected: {duplicates}')
if leaks: raise RuntimeError(f'Cross-split leakage detected: {len(leaks)} hashes')
print('No duplicate SHA256 values or cross-split leakage across',len(hash_splits),'files')

## 11. Model configuration

In [ ]:
CONFIG=Path('training/configs/indian_food_v2_candidate.json'); cfg=json.loads(CONFIG.read_text())
VERSION=os.getenv('ZENFIT_MODEL_VERSION','1.2.0-colab-candidate'); BASELINE_VERSION=os.getenv('ZENFIT_BASELINE_VERSION','1.1.0')
SMOKE_VERSION=VERSION+'-smoke'; SMOKE_ROOT=MODELS/SMOKE_VERSION; CANDIDATE=MODELS/VERSION
print(json.dumps(cfg,indent=2)); print({'smoke':str(SMOKE_ROOT),'candidate':str(CANDIDATE)})

## 12. Optional Google Drive mount

In [ ]:
MOUNT_DRIVE=False; BACKUP_AFTER_TRAINING=False
if MOUNT_DRIVE:
    if not IN_COLAB: raise RuntimeError('Google Drive mounting is only available in Colab')
    from google.colab import drive
    drive.mount('/content/drive'); DRIVE_ROOT=Path('/content/drive/MyDrive/ZenFit'); DRIVE_ROOT.mkdir(parents=True,exist_ok=True)
print('Drive root:',DRIVE_ROOT)

## 13. GPU smoke training

In [ ]:
RUN_GPU_SMOKE=False
def require_training_runtime():
    status=validate_prepared_dataset(DATASET)
    if not status['valid']: raise RuntimeError('Training blocked: '+'; '.join(status['errors']))
    if not torch.cuda.is_available(): raise RuntimeError('Training blocked: CUDA is unavailable')
    print('GPU:',torch.cuda.get_device_name(0)); return status['manifest']
def run_trainer(version,smoke):
    disk_manifest=require_training_runtime(); output=MODELS/version
    if output.exists(): raise FileExistsError(f'Immutable output already exists: {output}. Reuse it or choose a new VERSION.')
    cmd=[sys.executable,'training/train_indian_food.py',str(DATASET),'--models-dir',str(MODELS),'--config',str(CONFIG),'--version',version,'--dataset-version',disk_manifest['dataset_version'],'--device','cuda','--require-cuda','--num-workers','2']
    if smoke: cmd+=['--smoke','--smoke-samples-per-split','128']
    torch.cuda.reset_peak_memory_stats(); started=time.perf_counter(); subprocess.run(cmd,check=True); duration=time.perf_counter()-started
    metrics_path=output/'metrics.json'
    if not metrics_path.is_file(): raise RuntimeError(f'Trainer completed without {metrics_path}')
    result=json.loads(metrics_path.read_text()); print({'duration_seconds':round(duration,1),'peak_gpu_memory_gib':round(torch.cuda.max_memory_allocated()/2**30,3),'accuracy':result.get('accuracy'),'macro_f1':result.get('macro_f1'),'top_3_accuracy':result.get('top_3_accuracy')}); return output
if RUN_GPU_SMOKE: run_trainer(SMOKE_VERSION,True)
else: print('GPU smoke is disabled. Set RUN_GPU_SMOKE=True to run it manually.')

## 14. STOP — review smoke results manually
Do not enable full training until the smoke metrics and GPU usage above are acceptable.

In [ ]:
smoke_metrics_path=SMOKE_ROOT/'metrics.json'
if smoke_metrics_path.is_file(): print('Smoke result:',json.loads(smoke_metrics_path.read_text()))
else: print('Smoke training has not completed.')

## 15. Full training

In [ ]:
RUN_FULL_TRAINING=False
if RUN_FULL_TRAINING:
    if not smoke_metrics_path.is_file(): raise RuntimeError('Full training blocked: valid smoke metrics.json is missing')
    run_trainer(VERSION,False)
else: print('Full training is disabled. Set RUN_FULL_TRAINING=True only after reviewing the smoke run.')

## 16. Candidate validation

In [ ]:
def load_candidate_metrics(candidate):
    path=Path(candidate)/'metrics.json'
    if not path.is_file(): print('Candidate model not trained yet.'); return None
    try: return json.loads(path.read_text())
    except json.JSONDecodeError as exc: raise RuntimeError(f'Candidate metrics are invalid: {exc}')
metrics=load_candidate_metrics(CANDIDATE)
if metrics is not None: print({'best_epoch':metrics.get('best_epoch'),'best_validation_accuracy':metrics.get('best_validation_accuracy'),'checkpoint':metrics.get('checkpoint')})

## 17. Test metrics

In [ ]:
metrics=load_candidate_metrics(CANDIDATE)
if metrics is not None: print({k:metrics.get(k) for k in ('sample_count','accuracy','balanced_accuracy','macro_precision','macro_recall','macro_f1','top_3_accuracy')})

## 18. Confusion matrix

In [ ]:
metrics=load_candidate_metrics(CANDIDATE)
if metrics is not None:
    required=[CANDIDATE/'classes.json',CANDIDATE/'confusion_matrix.json']
    if not all(p.is_file() for p in required): raise RuntimeError('Candidate confusion-matrix artifacts are incomplete')
    import matplotlib.pyplot as plt, seaborn as sns
    classes=json.loads(required[0].read_text()); matrix=json.loads(required[1].read_text()); plt.figure(figsize=(12,10)); sns.heatmap(matrix,cmap='Blues',xticklabels=classes,yticklabels=classes); plt.tight_layout(); plt.savefig(REPORTS/f'{VERSION}-confusion-matrix.png',dpi=160); plt.show()

## 19. Per-class metrics

In [ ]:
metrics=load_candidate_metrics(CANDIDATE)
if metrics is not None: display(pandas.DataFrame(metrics['per_class']).T.sort_values('f1-score'))

## 20. Calibration

In [ ]:
metrics=load_candidate_metrics(CANDIDATE)
if metrics is not None:
    path=CANDIDATE/'calibration.json'
    if not path.is_file(): raise RuntimeError('Candidate calibration.json is missing')
    cal=json.loads(path.read_text()); print({k:cal.get(k) for k in ('temperature','ece','brier_score')})

## 21. Open-set evaluation

In [ ]:
OPEN_SET_EVIDENCE=REPORTS/f'{VERSION}-open-set-predictions.json'; OPEN_SET_REPORT=REPORTS/f'{VERSION}-open-set-evaluation.json'
metrics=load_candidate_metrics(CANDIDATE)
if metrics is None: print('Open-set evaluation skipped: candidate model not trained yet.')
elif not OPEN_SET_EVIDENCE.is_file(): print('Open-set evaluation skipped: evidence file is missing.')
else:
    thresholds=Path('training/configs/open_set_1.1.0.json'); subprocess.run([sys.executable,'training/evaluate_open_set.py',str(OPEN_SET_EVIDENCE),'--thresholds',str(thresholds),'--output',str(OPEN_SET_REPORT)],check=True); print('Experimental open-set report:',OPEN_SET_REPORT)
print('No open-set result is production approval.')

## 22. Threshold search

In [ ]:
THRESHOLD_REPORT=REPORTS/f'{VERSION}-threshold-report.json'; metrics=load_candidate_metrics(CANDIDATE)
if metrics is None: print('Threshold search skipped: candidate model not trained yet.')
elif not OPEN_SET_EVIDENCE.is_file(): print('Threshold search skipped: open-set evidence is missing.')
else:
    rows=json.loads(OPEN_SET_EVIDENCE.read_text()).get('predictions',json.loads(OPEN_SET_EVIDENCE.read_text()))
    from training.open_set_evaluation import threshold_sweep
    from training.analyze_open_set_thresholds import recommend
    recommendation=recommend(threshold_sweep(rows,VERSION,confidence_values=(.5,.57,.6,.65,.7),margin_values=(.05,.1,.15,.2),entropy_values=(None,1.0,1.5,2.0))); recommendation['status']='DEVELOPER_BETA_CANDIDATE'; THRESHOLD_REPORT.write_text(json.dumps(recommendation,indent=2)); (CANDIDATE/'open_set_thresholds.json').write_text(json.dumps(recommendation['thresholds'],indent=2)); print(json.dumps(recommendation,indent=2))

## 23. Candidate comparison

In [ ]:
if load_candidate_metrics(CANDIDATE) is not None: subprocess.run([sys.executable,'training/compare_indian_food_models.py',BASELINE_VERSION,VERSION,'--models-dir',str(MODELS)],check=False)

## 24. Developer-beta readiness

In [ ]:
metrics=load_candidate_metrics(CANDIDATE)
if metrics is None: print('Developer-beta readiness unavailable: candidate model not trained yet.')
else:
    from training.promote_indian_food import evaluate_gates
    gates=evaluate_gates(CANDIDATE); print(json.dumps(gates,indent=2)); print('Readiness only: this cell does not write developer_beta.json or active.json.')

## 25. Artifact export

In [ ]:
EXPORT_ARTIFACT=False; ARTIFACT=PACKAGES/VERSION
if EXPORT_ARTIFACT:
    required=['model.pt','classes.json','metrics.json','calibration.json','dataset_manifest.json','model_card.md']
    missing=[name for name in required if not (CANDIDATE/name).is_file()]
    if not THRESHOLD_REPORT.is_file(): missing.append(str(THRESHOLD_REPORT))
    if not (CANDIDATE/'open_set_thresholds.json').is_file(): missing.append('open_set_thresholds.json')
    if missing: raise RuntimeError('Artifact export blocked; missing: '+', '.join(missing))
    threshold_report=json.loads(THRESHOLD_REPORT.read_text()); thresholds=threshold_report.get('thresholds')
    if not isinstance(thresholds,dict): raise RuntimeError('Threshold report has no thresholds object')
    threshold_path=CANDIDATE/'open_set_thresholds.json'
    if json.loads(threshold_path.read_text())!=thresholds: raise RuntimeError('open_set_thresholds.json conflicts with threshold report')
    if ARTIFACT.exists(): raise FileExistsError(f'Refusing to overwrite artifact: {ARTIFACT}')
    subprocess.run([sys.executable,'scripts/package_model_artifact.py',str(CANDIDATE),str(ARTIFACT),'--environment','developer-beta'],check=True)
    from app.ai.artifacts import verify_artifact
    print(verify_artifact(ARTIFACT,required_environment='developer-beta'))
else: print('Artifact export is disabled.')

## 26. Independent inference smoke

In [ ]:
RUN_INFERENCE_SMOKE=False
if RUN_INFERENCE_SMOKE:
    from app.ai.artifacts import verify_artifact
    from app.ai.meal_scan.open_set import Candidate,OpenSetDecisionEngine,OpenSetInput,OpenSetThresholds,probability_entropy
    from training.train_indian_food import build_classifier_model,build_evaluation_transform
    verify_artifact(ARTIFACT,required_environment='developer-beta'); labels=json.loads((ARTIFACT/'classes.json').read_text()); config=json.loads((ARTIFACT/'config.json').read_text()); temperature=json.loads((ARTIFACT/'calibration.json').read_text())['temperature']; thresholds=OpenSetThresholds.from_json(ARTIFACT/'open_set_thresholds.json')
    exported=build_classifier_model(config['architecture'],len(labels),pretrained=False).cuda(); exported.load_state_dict(torch.load(ARTIFACT/'model.pt',map_location='cuda',weights_only=True)); exported.eval(); transform=build_evaluation_transform(config)
    groups={'known':list((DATASET/'test').glob('*/*'))[:5],'unknown':list((raw/'burger').glob('*'))[:3]+list((raw/'pizza').glob('*'))[:3],'non_food':list((LOCAL_ROOT/'open_set/non_food').glob('*'))[:6]}
    if any(not paths for paths in groups.values()): raise RuntimeError('Known, unknown, and non-food sample groups must all contain images')
    results=[]
    for group,paths in groups.items():
        for path in paths:
            image=Image.open(path).convert('RGB'); torch.cuda.synchronize(); started=time.perf_counter()
            with torch.inference_mode(): probs=(exported(transform(image).unsqueeze(0).cuda())/temperature).softmax(1)[0].cpu()
            torch.cuda.synchronize(); values,indices=probs.topk(min(3,len(labels))); top=tuple(Candidate(labels[int(i)],float(v)) for v,i in zip(values,indices)); decision=OpenSetDecisionEngine(thresholds).decide(OpenSetInput(top_candidates=top,entropy=probability_entropy(probs),model_version=VERSION)); row={'group':group,'file':path.name,'predicted_class':top[0].label,'confidence':top[0].confidence,'top_3':[{'label':x.label,'confidence':x.confidence} for x in top],'decision':decision.decision.value,'latency_ms':(time.perf_counter()-started)*1000}; results.append(row); print(row)
    (REPORTS/f'{VERSION}-independent-inference-smoke.json').write_text(json.dumps(results,indent=2)); del exported; torch.cuda.empty_cache()
else: print('Independent inference smoke is disabled.')

## 27. Optional backup/upload

In [ ]:
def backup_directory(source,category):
    if DRIVE_ROOT is None: raise RuntimeError('Mount Drive first')
    source=Path(source)
    if not source.exists(): print('Backup skipped; missing:',source); return
    destination=DRIVE_ROOT/'backups'/VERSION/category
    if destination.exists(): raise FileExistsError(f'Refusing to overwrite backup: {destination}')
    destination.parent.mkdir(parents=True,exist_ok=True); shutil.copytree(source,destination); print('Backed up:',destination)
if BACKUP_AFTER_TRAINING:
    backup_directory(SMOKE_ROOT,'smoke-model'); backup_directory(CANDIDATE,'candidate-model'); backup_directory(REPORTS,'reports'); backup_directory(ARTIFACT,'packaged-artifact')
else: print('Drive backup is disabled. Secrets are never copied to Drive.')

## 28. Final summary

In [ ]:
dataset_status=validate_prepared_dataset(DATASET); candidate_metrics=load_candidate_metrics(CANDIDATE)
print({'dataset_valid':dataset_status['valid'],'manifest_counts':dataset_status['manifest_counts'],'actual_counts':dataset_status['actual_counts'],'smoke_complete':(SMOKE_ROOT/'metrics.json').is_file(),'candidate_trained':candidate_metrics is not None,'artifact_exported':(ARTIFACT/'artifact_manifest.json').is_file(),'production_approved':False})
print('Next manual steps: validate/prepare dataset, confirm physical split folders, then enable RUN_GPU_SMOKE only.')